# Sparsity: Systematic Study

Top-k neuron sparsity per tick. Each experiment shows the ratio explicitly.

| group | what changes | runs |
|---|---|---|
| **Main** | +topk_neurons=0.5, 5 seeds x 2 tasks | 10 |
| **Sweep** | ratio ∈ {0.1..0.9}, 3 seeds x 2 tasks | 30 |

In [ ]:
import sys; sys.path.insert(0, '.'); sys.path.insert(0, '..')
from exp_runner import *
import matplotlib.pyplot as plt
%matplotlib inline

TASKS = ['sort', 'cifar10', 'mazes', 'parity']
SEEDS_MAIN = [0, 1, 2, 3, 4]
SEEDS_SWEEP = [0, 1, 2]

In [ ]:
for task in ['sort', 'mazes']:
    module, cfg = BASE_CONFIGS[task]
    print(f'{task:10s} d_model={cfg.get("d_model")}, iters={cfg.get("iterations")}')

## Prior Results (st08 sparsity0.5)

In [ ]:
df_prior = load_prior()
if df_prior is not None:
    print(summary_stats(df_prior[(df_prior.stage=='st08')&(df_prior.sweep=='sparsity0.5')]))
    plot_prior_bar(df_prior, ['sort','mazes'],
                   'st08', 'sparsity0.5', 'Prior: sparsity 0.5', 'figures/03_prior_bar.png')
else:
    print('Prior data not found.')

## Group 1 — Main (10 runs)

**Δ from baseline**:
```
+ topk_neurons = 0.5    # only top 50% neurons active per tick
```

That's it — one parameter. Sparsity is the simplest idea to add.

In [ ]:
exps_main = []
for task in ['sort', 'mazes']:
    module, base = BASE_CONFIGS[task]
    for s in [0,1,2,3,4]:
        exps_main.append(Experiment(
            name=f'{task}_sparsity0p5_s{s}',
            task=task, module=module,
            config={**base, 'seed': s,
                    'topk_neurons': 0.5}))    # + only top 50% active
print(f'{len(exps_main)} experiments')

## Group 2 — Ratio Sweep (30 runs)

**Only change**: `topk_neurons` ∈ {0.1, 0.25, 0.5, 0.75, 0.9}

0.1 = very sparse (10% active), 0.9 = almost dense.

In [ ]:
SWEEP_RATIOS = [0.1, 0.25, 0.5, 0.75, 0.9]
exps_sweep = []
for task in ['sort', 'mazes']:
    module, base = BASE_CONFIGS[task]
    for r in SWEEP_RATIOS:
        for s in [0,1,2]:
            exps_sweep.append(Experiment(
                name=f'{task}_swp_r{str(r).replace(".","p")}_s{s}',
                task=task, module=module,
                config={**base, 'seed': s,
                        'topk_neurons': r}))      # SWEEP: 0.1~0.9
print(f'{len(exps_sweep)} experiments  ({len(SWEEP_RATIOS)} ratios x 3 seeds x 2 tasks)')

## Run All

In [ ]:
exps = exps_main + exps_sweep
print(f'Total: {len(exps)}')
run_all(exps, gpus=8, log_root='logs/deep/03_sparsity', dry_run=True)

In [ ]:
done, failed = run_all(exps, gpus=8, log_root='logs/deep/03_sparsity')

In [ ]:
status('logs/deep/03_sparsity')

## Analysis

In [ ]:
df = collect('logs/deep/03_sparsity')
if df.empty:
    print('No results yet.')
else:
    df_main = df[df.name.str.contains('sparsity0p5_s') & ~df.name.str.contains('swp')]
    if not df_main.empty:
        print(df_main[['name','task','best_acc','delta']].to_string(index=False))
        plot_delta_bars(df_main, 'Sparsity(0.5) vs baseline', 'figures/03_main_delta.png')
        print(significance_test(df_main).to_string(index=False))

In [ ]:
if not df.empty:
    import re
    df_sw = df[df.name.str.contains('swp_')].copy()
    if not df_sw.empty:
        df_sw['ratio'] = df_sw['name'].str.extract(r'r([0-9]+p?[0-9]*)_s')[0].str.replace('p','.').astype(float)
        plot_sweep_heatmap(df_sw, 'ratio', 'task',
                          'Sparsity ratio x task delta (pp)', 'figures/03_sweep_heatmap.png')
        plot_sweep_curve(df_sw, 'ratio', 'Ratio sweep', 'figures/03_sweep_curve.png')